<div style="background: linear-gradient(120deg,#052e16,#14532d); border-radius:16px; padding:36px 40px; color:#e2e8f0;">
<h1 style="margin:0; font-size:2.1em; color:#ffffff;">🕸️ グラフデータベース</h1>
<h3 style="margin:6px 0 0 0; font-weight:400; color:#86efac;">セッション 3 / 4 — 知識には「形」がある</h3>
<p style="margin-top:18px; color:#cbd5e1;">NordWind Energy ワークショップシリーズ · 検索・ベクトル・グラフ</p>
</div>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noctetemp/nordwind-workshop/blob/main/session3_graphs_ja.ipynb)

## 🗺️ 今日の道のり

2 つのセッション分の証拠が、ひとつの結論を指しています：*空間上の点には「辺（エッジ）」がない。* チームのフィールドレポートがいちばん端的に言い当てていました——「ベクトル検索だけで正確な関係トラバーサルや網羅的な取得を行うのは不安定である」。今日、エッジは**一級市民**になります。

* **🔗 リレーショナルの痛み** — 「友達の友達の友達」がなぜ SQL を泣かせるのか：JOIN の爆発を、1 行で書けるグラフ版と並べて見ます。
* **✨ 生きているグラフ** — ここまで積み上げてきた瞬間：NordWind のすべて——全チーム、全エンジニア、全サービス、全インシデント——を、ドラッグできる物理シミュレーションのネットワークとして。誰かに説明される前に、自分で何かを*見つけて*しまうはずです。
* **🔌 Neo4j** — 自分のクラウドインスタンス（Aura）に接続し、数秒で世界をロードします。
* **🎨 Cypher** — 文字どおり ASCII アートなクエリ言語：`(:Engineer)-[:RESPONDED_TO]->(:Incident)`。あなたがパターンを描き、データベースがそれが現れるすべての場所を見つけます。
* **🪜 クエリのはしご** — 7 段のクエリ。「サービスを 1 件引く」から、SQL では本当に苦しいものまで。可変長パス、構造に対する集約、チーム横断の分析。
* **⚔️ ハードクエスチョン** — セッション 1 で素朴な RAG を壊し、セッション 2 で RAG v2 を生き延びたあの問い——**4 行で、根拠つきで、完全に**答えます。
* **🛰️ グラフナビゲーター** — 同じデータベースを、宇宙船の戦術ディスプレイとして：ノードをクリックすると、Neo4j が 3D で答えます。続けて **RAG の検死**：セッション 1 の誤答を地図の上に置き、ベクトルがなぜ失敗したのかを*目で見ます*。
* **🧠 いつグラフに手を伸ばすべきか** — そして、いつそうすべきでないか。モデリングの思考法：名詞はノードに、*動詞はリレーションシップに*。

> ⚠️ **事前準備：** 無料の Neo4j Aura インスタンス（3 分で作れます——リポジトリの `AURA_SETUP.md` を参照）。Aura がなくても、ビジュアルのセクションは動きますし、クエリのセルは隣の人とペアで進められます。

## 0 · セットアップ

In [ ]:
%pip -q install neo4j pyvis
print("✅ ready")

In [ ]:
import json, urllib.request

BASE = "https://raw.githubusercontent.com/noctetemp/nordwind-workshop/main/dataset/"
def fetch(path):
    return json.loads(urllib.request.urlopen(BASE + path).read())

teams     = fetch("entities/teams.json")
engineers = fetch("entities/engineers.json")
services  = fetch("entities/services.json")
incidents = fetch("entities/incidents.json")
rels      = fetch("relationships.json")
print(f"🌍 NordWind: {len(teams)} teams · {len(engineers)} engineers · "
      f"{len(services)} services · {len(incidents)} incidents · {len(rels)} relationships")

---
## 1 · 🔗 なぜ……SQL じゃダメなの？

もっともな疑問です——リレーショナルデータベースだって関係を*保存できます*。`service_dependencies(from_id, to_id)` というテーブルは、技術的にはグラフです。痛みが始まるのは、**トラバース（たどる）**ときです。

*「`payment-gateway` の障害は、最終的にどのサービスに波及しうるか？」*——依存元の、依存元の、依存元。SQL では：

```sql
-- 1 ホップ
SELECT s2.name FROM services s1
JOIN service_dependencies d1 ON d1.to_id = s1.id
JOIN services s2 ON s2.id = d1.from_id
WHERE s1.name = 'payment-gateway';

-- 2 ホップ：JOIN を JOIN する。3 ホップ：JOIN の JOIN を JOIN する。
-- 深さ不明：再帰 CTE、約 15 行。しかも各 JOIN はインデックス検索で、
-- そのコストはテーブルサイズとともに増えていく。
```

同じ問いを **Cypher** で、*どんな*深さでも：

```cypher
MATCH (down:Service)-[:DEPENDS_ON*1..]->(:Service {name: 'payment-gateway'})
RETURN DISTINCT down.name
```

これが単なる構文糖衣ではない、より深い理由が 2 つあります：

1. **インデックス不要の隣接（index-free adjacency）。** ネイティブなグラフストアでは、各ノードが隣接ノードへの直接ポインタを持っています。エッジを 1 つ渡ることはポインタ追跡であり、1 ホップあたりのコストは一定——*データベースの大きさに依存しません*。一方 JOIN はインデックス検索で、テーブルが育つほど高くつきます。
2. **リレーションシップが意味を運ぶ。** `DEPENDS_ON`、`OWNS`、`RESPONDED_TO` は型があり、向きがあり、プロパティも持てます——配管ではなく、データです。

しかし理論はここまで——2 セッションにわたって扱ってきたものを、まず*見る*べきです。

---
## 2 · ✨ 生きているグラフ

セッション 1 で集合演算にかけた、あの 153 本のリレーションシップ。今回は、物理シミュレーションつきで描きます。

🟡 六角形 = **チーム** · 🔵 丸 = **エンジニア** · 🟩 四角 = **サービス** · 🔴 ひし形 = **インシデント**

**ドラッグして。ズームして。ホバーして。** スクロールする前に、まる 2 分かけてください——この中に、自分なりの物語を見つけてください。

In [ ]:
from pyvis.network import Network
from IPython.display import HTML, display

COLORS = {"Team": "#f59e0b", "Engineer": "#60a5fa", "Service": "#34d399", "Incident": "#f87171"}

def build_net():
    net = Network(height="720px", width="100%", bgcolor="#0f172a",
                  font_color="#e2e8f0", directed=True, cdn_resources="remote")
    net.barnes_hut(gravity=-9000, central_gravity=0.25,
                   spring_length=140, spring_strength=0.02, damping=0.35)
    for t in teams:
        net.add_node(t["name"], label=t["name"], color=COLORS["Team"], shape="hexagon",
                     size=34, title=f"Team — {t['focus']}")
    for e in engineers:
        net.add_node(e["name"], label=e["name"].split()[0], color=COLORS["Engineer"],
                     size=16, title=f"{e['name']} — {e['role']}, {e['team_name']}")
    for s in services:
        net.add_node(s["name"], label=s["name"], color=COLORS["Service"], shape="box",
                     size=24, title=f"{s['description']} (owner: {s['owner_team']})")
    for i in incidents:
        net.add_node(i["id"], label=i["id"], color=COLORS["Incident"], shape="diamond",
                     size=20, title=f"{i['severity']} — {i['title']}")
    return net

EDGE_STYLE = {"MEMBER_OF": {"color": "#64748b", "width": 1},
              "OWNS": {"color": "#f59e0b", "width": 2},
              "DEPENDS_ON": {"color": "#34d399", "width": 2, "dashes": True},
              "AFFECTED": {"color": "#f87171", "width": 2},
              "RESPONDED_TO": {"color": "#93c5fd", "width": 1}}

net = build_net()
for r in rels:
    net.add_edge(r["from"], r["to"], title=r["type"], **EDGE_STYLE[r["type"]])

display(HTML(net.generate_html(notebook=False)))

🎤 **何を見つけましたか？** *（ファシリテーター向け：スクロールする前に「最初に直すべきサービスはどれ？」と会場に問いかけ、予想をひとつ選ばせてください。10 分後、クエリがその予想を採点します。）*

よく見つかるもの：billing-engine は赤い密林の中に座っています（実際に NordWind でもっともインシデントの多いサービスです——10 分後にクエリで証明します）。チームは、所有と所属に引っ張られて自然な「近所」を作ります。そして少数のエンジニアがクラスタの*間*に座っています——チーム横断の対応者たちです。

立ち止まって考える価値のある点：**この構造は、最初からドキュメントの中にありました。** セッション 1〜2 はこれらの事実をフラットなテキストとして読みました。今夜加わったのは*形*だけです。

では、これをクエリできるようにしましょう。

---
## 3 · 🔌 自分の Neo4j に接続する

Aura セットアップで得た認証情報（インスタンス作成時にダウンロードした `.txt` ファイル——`AURA_SETUP.md` を参照）を入力してください。URI は `neo4j+s://xxxxxxxx.databases.neo4j.io` のような形です。

In [ ]:
NEO4J_URI      = ""  # @param {type:"string"}
NEO4J_PASSWORD = ""  # @param {type:"string"}
NEO4J_USER     = "neo4j"

from neo4j import GraphDatabase
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()

def cypher(q, **params):
    """Run a Cypher query, return a list of dicts."""
    with driver.session() as s:
        return [r.data() for r in s.run(q, **params)]

print("🔌 connected:", cypher("RETURN 'hello from ' + 'Neo4j' AS msg")[0]["msg"])

世界をロードします：`UNWIND` は Python のリストを行に展開し、`MERGE` は各ノードがまだ存在しなければ作成します（再実行しても安全）。まずノード、次にリレーションシップ——リレーションシップは両端のノードが存在していなければ作れません：

In [ ]:
cypher("MATCH (n) DETACH DELETE n")   # clean slate — safe on your empty workshop instance

cypher("UNWIND $rows AS r MERGE (t:Team {name: r.name}) SET t.focus = r.focus", rows=teams)
cypher("UNWIND $rows AS r MERGE (e:Engineer {name: r.name}) SET e.role = r.role", rows=engineers)
cypher("UNWIND $rows AS r MERGE (s:Service {name: r.name}) "
       "SET s.description = r.description, s.language = r.language", rows=services)
cypher("UNWIND $rows AS r MERGE (i:Incident {id: r.id}) "
       "SET i.title = r.title, i.severity = r.severity, i.date = date(r.date)", rows=incidents)

by = lambda t: [r for r in rels if r["type"] == t]
cypher("UNWIND $rows AS r MATCH (e:Engineer {name: r.from}), (t:Team {name: r.to}) "
       "MERGE (e)-[:MEMBER_OF]->(t)", rows=by("MEMBER_OF"))
cypher("UNWIND $rows AS r MATCH (t:Team {name: r.from}), (s:Service {name: r.to}) "
       "MERGE (t)-[:OWNS]->(s)", rows=by("OWNS"))
cypher("UNWIND $rows AS r MATCH (a:Service {name: r.from}), (b:Service {name: r.to}) "
       "MERGE (a)-[:DEPENDS_ON]->(b)", rows=by("DEPENDS_ON"))
cypher("UNWIND $rows AS r MATCH (i:Incident {id: r.from}), (s:Service {name: r.to}) "
       "MERGE (i)-[:AFFECTED]->(s)", rows=by("AFFECTED"))
cypher("UNWIND $rows AS r MATCH (e:Engineer {name: r.from}), (i:Incident {id: r.to}) "
       "MERGE (e)-[:RESPONDED_TO]->(i)", rows=by("RESPONDED_TO"))

print(cypher("MATCH (n) RETURN count(n) AS nodes")[0],
      cypher("MATCH ()-[r]->() RETURN count(r) AS rels")[0])

**73 ノード、153 リレーションシップ。** 会社全体が 2 秒足らずでロードされました。💡 *Aura コンソール → Query タブでも `MATCH (n) RETURN n` を実行してみてください——Aura 組み込みの可視化で、データベースにライブ接続されたドラッグ可能なグラフが見られます。*

---
## 4 · 🎨 Cypher — 描けるクエリ

Cypher の核心：**探している形を描くと、データベースがそれが現れるすべての場所を見つけてくれる。**

```
(:Engineer)-[:RESPONDED_TO]->(:Incident)-[:AFFECTED]->(:Service)
   ノード ──────── エッジ ─────────► ノード ────── エッジ ──────► ノード
```

`()` がノード、`-[]->` がリレーションシップ、`{...}` でプロパティを固定。これで言語の 80% です。はしごを登りましょう——一段ごとに、ベクトル検索では一歩ずつ不可能になっていくクエリです：

In [ ]:
# Rung 1 — lookup (any database can do this)
cypher("MATCH (s:Service {name: 'payment-gateway'}) RETURN s.description AS description")

In [ ]:
# Rung 2 — one hop: who owns billing-engine?
cypher("MATCH (t:Team)-[:OWNS]->(s:Service {name: 'billing-engine'}) RETURN t.name AS owner")

In [ ]:
# Rung 3 — one hop + aggregation: the full Payments & Billing roster
cypher("""MATCH (e:Engineer)-[:MEMBER_OF]->(t:Team {name: 'Payments & Billing'})
          RETURN collect(e.name) AS roster""")

In [ ]:
# Rung 4 — blast radius: everything INC-2111 touched
cypher("""MATCH (i:Incident {id: 'INC-2111'})-[:AFFECTED]->(s:Service)
          RETURN i.title AS incident, collect(s.name) AS affected_services""")

In [ ]:
# Rung 5 — VARIABLE-LENGTH paths: everything customer-portal transitively depends on.
# *1..3 means "follow 1 to 3 DEPENDS_ON edges" — this is the recursive-CTE killer.
cypher("""MATCH (:Service {name: 'customer-portal'})-[:DEPENDS_ON*1..3]->(up:Service)
          RETURN collect(DISTINCT up.name) AS upstream_dependencies""")

Rung 5 の結果をよく読んでください：customer-portal は、billing-engine を経由して **payment-gateway**、さらには **meter-reader** にまで推移的に依存しています——3 ホップ先です。これを述べたドキュメントはどこにもありません。*構造から導出された*のです。チームのフィールドレポートが求めていたのは、まさにこれ——漏れのない、網羅的な関係探索です。

In [ ]:
# Rung 6 — aggregation over structure: NordWind's most incident-prone services
cypher("""MATCH (i:Incident)-[:AFFECTED]->(s:Service)
          RETURN s.name AS service, count(i) AS incident_count
          ORDER BY incident_count DESC LIMIT 5""")

In [ ]:
# Rung 7 — structural analysis: incidents that required responders from MULTIPLE teams
# (a proxy for organizational complexity — pure graph thinking)
cypher("""MATCH (e:Engineer)-[:MEMBER_OF]->(t:Team), (e)-[:RESPONDED_TO]->(i:Incident)
          WITH i, count(DISTINCT t) AS teams_involved
          WHERE teams_involved > 1
          RETURN count(i) AS cross_team_incidents""")

20 件中 13 件のインシデントが複数チームを必要としました。ベクトルインデックスにこれを聞いてみてください。😄

*数えることと網羅性*——フィールドレポートが「信頼できない」と指摘した 2 つ——に何が起きたか注目してください。グラフに対する `count`、`collect`、`DISTINCT` は**正確**です。「最も関連性の高い結果」ではなく、*その*結果。すべて、証明可能に。

---
## 5 · ⚔️ ハードクエスチョン — 回収の時間

3 セッション前、この問いは素朴な RAG を壊しました。前回は、ハイブリッド検索と*リランキング*を生き延びました。必要なのは：依存関係の知識 → インシデントとの結びつき → 対応者の網羅的な集約。3 種類のリレーションシップにまたがる結合です。

Cypher では、ただ……描くだけです：

In [ ]:
cypher("""
MATCH (dep:Service)-[:DEPENDS_ON]->(:Service {name: 'payment-gateway'})
MATCH (i:Incident)-[:AFFECTED]->(dep)
MATCH (e:Engineer)-[:RESPONDED_TO]->(i)
WITH e, i ORDER BY e.name
RETURN collect(DISTINCT e.name) AS engineers,
       count(DISTINCT i)        AS qualifying_incidents
""")

**10 人のエンジニア全員。6 件のインシデント全部。4 行。ミリ秒。決定的**——1000 回実行すれば 1000 回同じ答えが返ります。類似度ランキングには決して約束できないことです。

そして RAG の自信たっぷりな部分的回答とは違い、この答えは*根拠を示せます*。あるエンジニアが**なぜ**該当するのか、グラフに聞いてみましょう：

In [ ]:
cypher("""
MATCH p = (e:Engineer {name: 'Winry Rockbell'})-[:RESPONDED_TO]->(:Incident)
          -[:AFFECTED]->(:Service)-[:DEPENDS_ON]->(:Service {name: 'payment-gateway'})
RETURN [n IN nodes(p) | coalesce(n.name, n.id)] AS evidence_path
""")

`Winry Rockbell → INC-2105 → billing-engine → payment-gateway`——**推論が添えられた答え**です。規制のある領域や安全に関わる文脈では、この説明可能性は「あれば嬉しい」ではなく、要件そのものです。

では答えを*見ましょう*——同じ生きているグラフに、すべての根拠パスを点灯させます：

In [ ]:
# recompute the answer set in Python (same 3 hops, set-style) to drive the highlight
dependents  = {r["from"] for r in rels if r["type"] == "DEPENDS_ON" and r["to"] == "payment-gateway"}
qual_incs   = {r["from"] for r in rels if r["type"] == "AFFECTED" and r["to"] in dependents}
answer_engs = {r["from"] for r in rels if r["type"] == "RESPONDED_TO" and r["to"] in qual_incs}
lit = answer_engs | qual_incs | dependents | {"payment-gateway"}

net2 = build_net()
for n in net2.nodes:                       # dim everything...
    if n["id"] not in lit:
        n["color"] = "#334155"; n["opacity"] = 0.25
    else:                                  # ...then relight the answer
        n["size"] = n.get("size", 16) + 10
        n["borderWidth"] = 3

def on_path(r):
    return ((r["type"] == "RESPONDED_TO" and r["from"] in answer_engs and r["to"] in qual_incs) or
            (r["type"] == "AFFECTED"     and r["from"] in qual_incs   and r["to"] in dependents) or
            (r["type"] == "DEPENDS_ON"   and r["from"] in dependents  and r["to"] == "payment-gateway"))

for r in rels:
    if on_path(r):
        net2.add_edge(r["from"], r["to"], color="#fbbf24", width=4, title=r["type"])
    else:
        net2.add_edge(r["from"], r["to"], color="#1e293b", width=1)

display(HTML(net2.generate_html(notebook=False)))

**あの金色の網が答えです。** 10 人のエンジニア、6 件のインシデント、1 本の依存エッジ——関係のなかったすべてのノイズの中に、光で描かれた結合。この絵こそ「構造による検索」の*意味*であり、誰かに「なぜグラフ？」と聞かれたときに思い出すべき絵です。

---
## 5b · 🛰️ グラフナビゲーター — データベースが答える瞬間を見る

ここまではすべてテーブルが返ってきました。今度は同じデータベースを、宇宙船の戦術ディスプレイとして：

**👉 [グラフナビゲーターを開く](https://raw.githack.com/noctetemp/nordwind-workshop/main/nordwind_3d.html)** — **🔌 UPLINK** を押し、セクション 3 で使った*同じ* URI とパスワードを貼り付けてください。このページは**データを一切持っていません**：これから見えるすべての球体は、数ミリ秒前に届いた Cypher の結果です（下部のログに、各クエリのレコード数と所要時間が表示されます）。

虚空にぽつんと、緑のノードが 1 つ現れます——`payment-gateway`。では**ナビゲート**しましょう：

1. **クリックする。** ナビゲーターがそのノードに対して `MATCH (n)-[r]-(m)` を実行し、隣接ノードが実体化します。その中に billing-engine がいます。
2. **billing-engine をクリック。** customer-portal が現れます。
3. ここで一呼吸。*customer-portal が payment-gateway に依存していると書いたドキュメントは、どこにもありません。* あなたはエッジを 2 本歩いて、それを導出しました——**クリック = ホップ**。「トラバーサル」とはただそれだけのことであり、Rung 5 が 1 行でやったことです。
4. 会場からリクエストを取ってください——*「Levi Ackerman を見せて」*——そしてコンソールに打ち込みます：
   `MATCH (e:Engineer {name:'Levi Ackerman'})-[r]-(m) RETURN e,r,m`
5. **⚡ TRACE IMPACT** を押す。ハードクエスチョンが本物のパスクエリとして実行され、前のセルの金色の網が 3 次元で現れます——あなたのデータベースからライブで引き出されて。

**さきほどの予想**——最初に直すべきサービスは？ **⬇ LOAD ALL** を押し、回転させながら赤い密林を見てください。次に Rung 6 のテーブルを見てください。直感とクエリは、*ここでは*一致します。次のセルでは一致しません——そのことを覚えておいてください。

### 🔬 RAG の検死

セッション 1 で、素朴な RAG はハードクエスチョンに **INC-2117、INC-2107、INC-2113** と答えました——自信たっぷりに、そして間違って。私たちは「決済っぽく*聞こえる*インシデントを取ってきた」と言いました。今なら比喩よりうまくやれます：誤答を*地図の上に置く*ことができます。

問いが求めているのは、payment-gateway に**依存する**サービス上のインシデントです。RAG の 3 つの選択が該当するかどうか、グラフに聞いてみましょう：

In [ ]:
cypher("""
MATCH (i:Incident) WHERE i.id IN ['INC-2117', 'INC-2107', 'INC-2113']
RETURN i.id  AS incident,
       [(i)-[:AFFECTED]->(x) | x.name]  AS affected_services,
       EXISTS { (i)-[:AFFECTED]->(:Service)-[:DEPENDS_ON]->(:Service {name: 'payment-gateway'}) }
                                          AS qualifies
ORDER BY incident
""")

3 行、3 つの `false`——そして*理由を見てください*：

- **INC-2117 と INC-2113 は `payment-gateway` そのものに当たっています。** 問いの*主題*についてのインシデントであって、*答え*ではありません。問いが求めているのは 1 ホップ*上流*のサービスです——ベクトルには「X について」と「X につながっている」の区別がつきませんでした。エッジの向きを符号化する埋め込みは存在しません。
- **INC-2107 は `api-gateway` に当たっています。** まったく別のゲートウェイです。「gateway」という単語だけで引き寄せられた——関連性を装った字面の類似です。

では、会場に*見せましょう*。ナビゲーターで TRACE を点灯させたまま、コンソールにこれを貼り付けます：

```cypher
MATCH (i:Incident) WHERE i.id IN ['INC-2117','INC-2107','INC-2113'] MATCH (i)-[r]-(m) RETURN i, r, m
```

3 つのうち 2 つは payment-gateway に*触れて*点灯します——ハブに隣接しているのに、**どの金色のパスにも乗っていない**。3 つ目は別のゲートウェイのそばに浮いています。RAG は*問いに似たテキスト*を見つけ、グラフは*答えにつながるノード*を見つけました。この 1 枚の絵が、セッション 1〜3 の主張のすべてです。

> 🧭 **きれいな絵についての告白。** 73 ノードなら圧巻です。70,000 ノードでは毛玉になり、目は役に立ちません——billing-engine の密林も、3 つの偽インシデントも見つけられないでしょう。これはツールの弱点ではなく、*クエリ言語が存在する理由*です。見ることは**形**の問い（ハブはどこ？クラスタは何？）に答えます。**リスト**の問い（正確にどの 10 人？）に答えられるのはクエリだけです。絵で直感を育て、Cypher で正しくあってください。

> 🔭 **いま、マウスで何をしたか気づいてください。** *名前のある*ノードから始めて、数ホップ外へ広げ、つながっているものを集めました。この手つきを覚えておいてください。次回、ベクトル検索が開始ノードを選び（「どこから始めるか」）、グラフが展開を行います（「何がつながっているか」）——それが GraphRAG アルゴリズムのすべてです。あなたはすでに手作業でそれを実行しました。

---
## 6 · 🧠 いつグラフに手を伸ばすべきか — そして、いつそうすべきでないか

**モデリングの思考法：** ドメインの言葉に耳を傾けてください。**名詞はノードになります**（Engineer、Service、Incident）。**動詞はリレーションシップになります**（owns、depends on、responded to）。ステークホルダーが自然に「X は Z を通じて Y に*つながっている*」と言うなら、あなたはスキーマを教えてもらっているのです。

| グラフに手を伸ばすとき…… | テーブル / ベクトルにとどまるとき…… |
|---|---|
| 問いが「たどる」：「〜を通じて」「〜につながる」「〜の影響」「誰が誰と働いたか」 | 問いがフラットなレコードをフィルタし集約する：「地域別の売上」 |
| 深さが不明または可変（`*1..`） | 結合が浅く固定（既知の 1〜2 ホップ） |
| 網羅性と説明可能性が要件 | 「良いマッチ」で十分 |
| リレーションシップこそがデータ（組織図、依存関係、不正リング、リネージ） | リレーションシップは付随的な外部キー |
| — | 内容が非構造化の散文 → それこそ**ベクトル**の出番 |

最後の行が最重要です：**グラフとベクトルは競合ではありません。** グラフは*誰が何につながっているか*を知り、ベクトルは*ドキュメントに何が書いてあるか*を知っています。NordWind のグラフは INC-2105 の対応者を挙げられますが、ポストモーテムの散文は読んでいません。ベクトルインデックスはすべてを読んでいますが、構造は見えていません。両方が必要です。

## 🏁 今日学んだこと

- たどる形の問いは SQL を悲鳴させ、Cypher を肩すくめさせる——インデックス不要の隣接がその理由
- Cypher = パターンを描いて、すべての出現を得る：`MATCH`、プロパティ、`*1..n` パス、正確な集約
- ハードクエスチョンは 4 行で落ちた——**根拠パスつきで**——決定的、網羅的、説明可能
- 名詞 → ノード、動詞 → リレーションシップ；構造にはグラフ、散文にはベクトル
- 絵は**形**を見せ、クエリが**リスト**を答える——RAG の誤答は地図の上でハブに触れていたが、パスには乗っていなかった

## 📝 次回までに
練習用ノートブックには NordWind グラフでの Cypher 型（かた）があります——その中に、グラフ*でも*ベクトル*でも*単独では答えられない問いが 1 つ隠れています。それを見つけられたら、セッション 4 が始まる前に理解したことになります。

---

<div style="background: linear-gradient(120deg,#450a0a,#7f1d1d); border-radius:16px; padding:30px 36px; color:#e2e8f0;">
<h2 style="margin:0; color:#ffffff;">⏭️ つづく……</h2>
<p style="font-size:1.05em; margin-top:14px;">今夜の魔法は、ひとつの告白の上で動いていました：私たちのグラフは、架空の世界に<i>付属してきた</i>きれいな <code>relationships.json</code> からロードされたのです。あなたの本物の会社には、そんなファイルはありません。依存関係は ADR の散文の中に、対応者リストはポストモーテムの中に、組織図は人々の頭の中にあります。</p>
<p style="font-size:1.05em;">次回——最終回：LLM が NordWind の生のドキュメントを読み、<b>このグラフを自分で組み立てます</b>。そしてすべてを融合します：ベクトルが<i>どこから始めるか</i>を見つけ、グラフが<i>何がつながっているか</i>を見つけ、このワークショップを始めた問いに、散文だけから端から端まで答えます。</p>
<h3 style="color:#fca5a5; margin-bottom:0;">セッション 4: GraphRAG — <i>一巡して元へ</i> 🔄</h3>
</div>